# Pipeline de limpieza de ESCO

El primer paso es importar las librerias que se van a utilizar.

In [1]:
from pathlib import Path
import pandas as pd
pd.set_option("display.max_colwidth", None)
pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)

## Carga de datos

In [ ]:
ruta_datos = Path("../data/raw/esco") # Definir la ruta de los archivos

occupations_original = pd.read_csv(ruta_datos / "occupations_es.csv", encoding="utf-8")
skills_original = pd.read_csv(ruta_datos / "skills_es.csv", encoding="utf-8")
occupation_skill_relations_original = pd.read_csv(ruta_datos / "occupationSkillRelations_es.csv", encoding="utf-8")

# Copias de trabajo para aplicar las transformaciones
occupations = occupations_original.copy()
skills = skills_original.copy()
occupation_skill_relations = occupation_skill_relations_original.copy()

## Transformaciones

### Fechas

La columna `modifiedDate` de los conjuntos de datos `occupations` y `skills` contiene fechas almacenadas como cadenas de texto. Se convierten al tipo datetime para facilitar su tratamiento y permitir operaciones cronológicas.

Se utiliza el formato ISO 8601 y se establece UTC como zona horaria.

In [3]:
for nombre, df in {"occupations": occupations, "skills": skills}.items():
    df["modifiedDate"] = pd.to_datetime(df["modifiedDate"], format="ISO8601", errors="raise", utc=True)

### Espacios sobrantes

Se detectaron espacios en blanco sobrantes al principio o al final de algunos valores de texto. Se corrige ahora esta inconsistencia.

In [4]:
# Columnas afectadas de occupations
occupations["description"] = (occupations["description"].str.strip())

# Columnas afectadas de skills
skills["preferredLabel"] = (skills["preferredLabel"].str.strip())

skills["description"] = (skills["description"].str.strip())

### Valores duplicados

Durante el análisis exploratorio se detectaron URI repetidas en los conjuntos de datos `occupations` y `skills`. En ambos casos, los registros asociados a una misma URI presentaban diferencias únicamente en la columna `modifiedDate`.

Se decide conservar el registro con la fecha de modificación más reciente para cada URI y eliminar las versiones anteriores, evitando mantener distintas versiones del mismo concepto.

In [5]:
# Ordenar por fecha de modificación, de más reciente a más antigua
occupations = occupations.sort_values(by="modifiedDate", ascending=False)
skills = skills.sort_values(by="modifiedDate", ascending=False)

# Conservar únicamente el registro más reciente de cada URI
occupations = occupations.drop_duplicates(subset="conceptUri", keep="first").reset_index(drop=True)
skills = skills.drop_duplicates(subset="conceptUri", keep="first").reset_index(drop=True)

De otra forma, se vió que en `skills` habían 3 registros con el campo `preferredLabel` duplicado que no tenían relaciones. Se decide eliminar esos registros puesto que no aportan información.

In [6]:
# Identificar las URI de skills que aparecen en las relaciones
skills_con_relaciones = occupation_skill_relations["skillUri"].unique()

# Denominaciones que queremos revisar
labels_duplicadas = [
    "asumir la responsabilidad",
    "calcular probabilidades",
    "improvisar"
]

# Filtrar los registros de esas denominaciones que no tienen relaciones
mascara = (
    skills["preferredLabel"].isin(labels_duplicadas)
    & ~skills["conceptUri"].isin(skills_con_relaciones)
)

# Eliminar los registros
skills = skills[~mascara].reset_index(drop=True)


### Valores ausentes

Se identificaron las columnas `scopeNote` y `definition` con un 100 % de valores ausentes en los conjuntos de datos analizados.

Al no contener información en ninguno de sus registros, se decide eliminar ambas columnas para evitar conservar variables que no aportan información útil.

In [7]:
columnas_eliminar = ["scopeNote", "definition"]

occupations = occupations.drop(columns=columnas_eliminar)
skills = skills.drop(columns=columnas_eliminar)

Después se tratan los valores ausentes encontrados en `skillType` y `reuseLevel` en `skills`. Como esos cinco registros formaban parten de las competencias digiComp, y los otros registros de las mismas competencias con dichos valores no nulos tenían todos el mismo valor, se decide rellenar estos con ese mismo valor.

In [8]:
# Identificar registros con valores ausentes
mascara = skills["skillType"].isna() | skills["reuseLevel"].isna()

# Rellenar los valores ausentes
skills.loc[mascara, "skillType"] = (skills.loc[mascara, "skillType"].fillna("skill/competence"))

skills.loc[mascara, "reuseLevel"] = (skills.loc[mascara, "reuseLevel"].fillna("cross-sector"))

In [ ]:
# Crear un diccionario que relaciona cada skill_uri con su skill_type
diccionario_skill_type = skills.set_index("conceptUri")["skillType"]

# Completar los valores ausentes en occupation_skill_relations
mascara = occupation_skill_relations["skillType"].isna()

occupation_skill_relations.loc[mascara, "skillType"] = (
    occupation_skill_relations.loc[mascara, "skillUri"]
    .map(diccionario_skill_type)
)

Valores ausentes en skill_type: 0


Por otras parte, se decide combinar las columnas `altLabels` y `hiddenLabels` en una única columna `altLabels`, ya que ambas contienen denominaciones alternativas. De esta manera, se centralizan en un solo campo y se elimina la columna `hiddenLabels`, evitando mantener información separada con la misma finalidad.

In [10]:
for nombre in ["occupations", "skills"]:
    df = globals()[nombre]

    # Combinar las denominaciones alternativas, respetando los saltos de línea
    df["altLabels"] = (
        df["altLabels"].fillna("").str.strip()
        + "\n"
        + df["hiddenLabels"].fillna("").str.strip()
    ).str.strip().replace("", pd.NA)

    # Eliminar la columna hiddenLabels
    df = df.drop(columns="hiddenLabels")

    globals()[nombre] = df

### Columnas sin información

Se vio también que habían columnas en `occupations` y en `skills` que tan solo repetían un mismo valor en todos los registros, `conceptType` y `status`. Como no aportan información, se decide deshacerse de las mismas.

Por otra parte, tras hacer la limpieza, la columna `modifiedDate` ya no aporta información necesaria.

In [11]:
occupations = occupations.drop(columns=["conceptType", "status", "modifiedDate"])
skills = skills.drop(columns=["conceptType", "status", "modifiedDate"])

### Formato nombre columnas

Se cambia el nombre de las columnas para que estén en snake_case.

In [12]:
for df in [occupations, skills, occupation_skill_relations]:
    df.columns = (
        df.columns
        .str.replace(r"([a-z0-9])([A-Z])", r"\1_\2", regex=True)
        .str.lower()
    )

## Validación

### Nombres columnas

Primero se comprueba que se ha cambiado correctamente los nombres de las columnas, porque es lo que se va a utilizar para llamarlas.

In [13]:
for nombre, df in {
    "occupations": occupations,
    "skills": skills,
    "occupation_skill_relations": occupation_skill_relations
}.items():
    print(f"\n{nombre}:")
    print(df.columns.tolist())


occupations:
['concept_uri', 'isco_group', 'preferred_label', 'alt_labels', 'regulated_profession_note', 'in_scheme', 'description', 'code', 'nace_code']

skills:
['concept_uri', 'skill_type', 'reuse_level', 'preferred_label', 'alt_labels', 'in_scheme', 'description']

occupation_skill_relations:
['occupation_uri', 'occupation_label', 'relation_type', 'skill_type', 'skill_uri', 'skill_label']


### Espacios sobrantes

Se comprueba que ya no quedan espacios sobrantes.

In [14]:
columnas_a_comprobar = {
    "occupations": {
        "df": occupations,
        "columnas": ["description"]
    },
    "skills": {
        "df": skills,
        "columnas": ["preferred_label", "description"]
    }
}

for nombre, datos in columnas_a_comprobar.items():
    print(f"\n{nombre}")
    print("-" * 50)

    for columna in datos["columnas"]:
        valores = datos["df"][columna].dropna().astype(str)
        espacios = valores.str.strip().ne(valores).sum()

        print(f"{columna}: {espacios} registros con espacios sobrantes")


occupations
--------------------------------------------------
description: 0 registros con espacios sobrantes

skills
--------------------------------------------------
preferred_label: 0 registros con espacios sobrantes
description: 0 registros con espacios sobrantes


### Valores ausentes

Se comprueba que, al eliminar columnas, han cambiado las dimensiones de las muestras.

In [15]:
for nombre, df in {"occupations": occupations, "skills": skills}.items():
    print(f"\n{nombre}")
    print("-" * 40)
    print("Dimensiones:", df.shape)
    # columnas occupations: 15 - 2 vacias - hiddenLabels - 2 de un solo valor - modifiedDate
    # columnas skills: 13 - 2 vacias - hiddenLabels - 2 de un solo valor - modifiedDate


occupations
----------------------------------------
Dimensiones: (3039, 9)

skills
----------------------------------------
Dimensiones: (13936, 7)


Luego se comprueba que ya no queda valores ausentes en las columnas `skillType` y `reuseLevel` de `skills`.

In [ ]:
print("Valores ausentes en skill_type de skills:", skills["skill_type"].isna().sum())
print("Valores ausentes en reuse_level de skills:", skills["reuse_level"].isna().sum())
print(f"Valores ausentes en skill_type de occupation_skill_relations: {occupation_skill_relations["skill_type"].isna().sum()}")

Valores ausentes en skill_type de skills: 0
Valores ausentes en reuse_level de skills: 0
Valores ausentes en skill_type de occupation_skill_relations: 0


### Valores duplicados

Se comprueba que ya no quedan Uris repetidos.

In [17]:
for nombre, df, df_original in [
    ("occupations", occupations, occupations_original),
    ("skills", skills, skills_original)
]:

    registros_eliminados = len(df_original) - len(df)
    uri_duplicadas = df["concept_uri"].duplicated().sum()

    print(f"\n{nombre}")
    print("-" * 45)
    print(f"Registros iniciales: {len(df_original):,}")
    print(f"Registros finales: {len(df):,}")
    print(f"Registros eliminados: {registros_eliminados}")
    print(f"URI duplicadas restantes: {uri_duplicadas}")


occupations
---------------------------------------------
Registros iniciales: 3,043
Registros finales: 3,039
Registros eliminados: 4
URI duplicadas restantes: 0

skills
---------------------------------------------
Registros iniciales: 13,960
Registros finales: 13,936
Registros eliminados: 24
URI duplicadas restantes: 0


Se comprueban luego los duplicados que han quedado en `preferredLabel` de `skills` tras eliminar algunos registros.

In [18]:
print("Duplicados por conceptUri:", skills["concept_uri"].duplicated().sum())
print("Duplicados por preferredLabel:", skills["preferred_label"].duplicated().sum()) #41 iniciales - 21 Uris repetidos - 3 sin relaciones = 17 
print("Registros completamente duplicados:", skills.duplicated().sum())

Duplicados por conceptUri: 0
Duplicados por preferredLabel: 17
Registros completamente duplicados: 0


Despues de haber eliminado registros, se comprueba que no se ha eliminado ninguno que no correspondía. Es decir, se comprueba que todos los Uris en occupationSkillRelations sigue correspondiendose con un Uri de occupations o de skills.

In [19]:
# Comprobar URI de ocupaciones
uris_occupations = set(occupations["concept_uri"])
uris_relaciones_occupations = set(occupation_skill_relations["occupation_uri"])

ocupaciones_sin_correspondencia = uris_relaciones_occupations - uris_occupations


# Comprobar URI de competencias
uris_skills = set(skills["concept_uri"])
uris_relaciones_skills = set(occupation_skill_relations["skill_uri"])

skills_sin_correspondencia = uris_relaciones_skills - uris_skills


# Mostrar resultados
print("URI de ocupaciones en relaciones sin correspondencia:", len(ocupaciones_sin_correspondencia))
print("URI de competencias en relaciones sin correspondencia:", len(skills_sin_correspondencia))

URI de ocupaciones en relaciones sin correspondencia: 0
URI de competencias en relaciones sin correspondencia: 0


### Tipos de datos

Finalmente se comprueba que no haya habido ningún error, cambiando tipos de datos sin querer.

In [23]:
# Revisar tipos de datos de los tres DataFrames
for nombre, df in {
    "occupations": occupations,
    "skills": skills,
    "occupation_skill_relations": occupation_skill_relations
}.items():

    print(f"\n{nombre}")
    print("-" * 60)
    df.info()


occupations
------------------------------------------------------------
<class 'pandas.DataFrame'>
RangeIndex: 3039 entries, 0 to 3038
Data columns (total 9 columns):
 #   Column                     Non-Null Count  Dtype
---  ------                     --------------  -----
 0   concept_uri                3039 non-null   str  
 1   isco_group                 3039 non-null   int64
 2   preferred_label            3039 non-null   str  
 3   alt_labels                 2899 non-null   str  
 4   regulated_profession_note  3039 non-null   str  
 5   in_scheme                  3039 non-null   str  
 6   description                3039 non-null   str  
 7   code                       3039 non-null   str  
 8   nace_code                  3039 non-null   str  
dtypes: int64(1), str(8)
memory usage: 213.8 KB

skills
------------------------------------------------------------
<class 'pandas.DataFrame'>
RangeIndex: 13936 entries, 0 to 13935
Data columns (total 7 columns):
 #   Column           N

## Salida

Finalmente, se guardan los cambios de la capa silver.

In [24]:
ruta_silver = Path("../data/silver/esco") # Definir la ruta de destino

# Crear la carpeta si todavía no existe
ruta_silver.mkdir(parents=True, exist_ok=True)

# Guardar los DataFrames limpios
occupations.to_csv(ruta_silver / "occupations_es_silver.csv", index=False, encoding="utf-8")
skills.to_csv(ruta_silver / "skills_es_silver.csv", index=False, encoding="utf-8")
occupation_skill_relations.to_csv(ruta_silver / "occupation_skill_relations_es_silver.csv", index=False, encoding="utf-8")

print("Los tres archivos se han guardado correctamente.")

Los tres archivos se han guardado correctamente.
